In [1]:
from IPython.display import display
import pandas as pd
import numpy as np

import main_analysis as ma

In [2]:
probe_csv = pd.read_csv(ma.BEST_LAYERS_CSV_PATH)
crossval_csv = pd.read_csv(ma.MODELSUBJECT_CORRELATIONS_CSV_PATH)
froi_fmri_model_map = {
    # fmri fROI : model fROI
    'FFA': 'FFA',
    'OFA': 'FFA',
    'PPA': 'PPA',
    'OPA': 'PPA',
    'RSC': 'PPA',
    'EBA': 'EBA',
    'FBA': 'EBA',
}
frois = ['FFA', 'PPA', 'EBA']
tthreshold = 7
model_localizer = 'grillspector'

models_df = []  # List to hold DataFrames for each fROI

# Median across fROIs

In [10]:
probe_csv_selected = probe_csv[
    (probe_csv['fmri_froi'].isin(frois)) &
    (probe_csv['model_froi'].isin(frois)) &
    (probe_csv['fmri_tthreshold'] == tthreshold) &
    (probe_csv['model_tthreshold'] == tthreshold) &
    (probe_csv['model_localizer'] == model_localizer)
]
# Remove the row corresponding to model name resnet50imagenet21k
probe_csv_selected = probe_csv_selected[probe_csv_selected['model_name'] != 'resnet50imagenet21k']
# Group by the model_name and take the median of normed_pearsonr_median, ignore non-matching layers
# probe_csv_selected = probe_csv_selected.drop(columns=['model_layer'])
probe_csv_selected = probe_csv_selected.groupby('model_name')['normed_pearsonr_median'].median().reset_index()
# Sort the DataFrame by normed_pearsonr_median in descending order, for visualization purposes
probe_csv_selected = probe_csv_selected.sort_values(by='normed_pearsonr_median', ascending=False).reset_index(drop=True)
# Get the best, median, and worst model for this fROI based on normed_pearsonr_median
best_model_info = probe_csv_selected.loc[probe_csv_selected['normed_pearsonr_median'].idxmax()]
worst_model_info = probe_csv_selected.loc[probe_csv_selected['normed_pearsonr_median'].idxmin()]
# Find the median model using argsort
sorted_indices = np.argsort(probe_csv_selected['normed_pearsonr_median'].values)
median_idx = len(sorted_indices) // 2
median_model_info = probe_csv_selected.iloc[sorted_indices[median_idx]]

# Get the top 10 models
print("Model ranks based on normed_pearsonr_median:")
top_indices = np.argsort(probe_csv_selected['normed_pearsonr_median'].values)[::-1]
top_models = probe_csv_selected.iloc[top_indices]
display(top_models)

# Store the results in a DataFrame
models_df = [
    {
        'model_type': 'best',
        'model_name': best_model_info['model_name'],
        'probe_normed_pearsonr_median': best_model_info['normed_pearsonr_median'],
    }, 
    {
        'model_type': 'median',
        'model_name': median_model_info['model_name'],
        'probe_normed_pearsonr_median': median_model_info['normed_pearsonr_median'],
    },
    {
        'model_type': 'worst',
        'model_name': worst_model_info['model_name'],
        'probe_normed_pearsonr_median': worst_model_info['normed_pearsonr_median'],
    }
]
models_df = pd.DataFrame(models_df)
display(models_df)

Model ranks based on normed_pearsonr_median:


,model_name,normed_pearsonr_median
0,vitb16siglip2res224,0.843869
1,resnet50openaiclip,0.838861
2,convnexttinyimagenet1kv1,0.837264
3,resnet101openaiclip,0.837051
4,vitb14dinov2reg,0.831582
5,resnet50simclr,0.817388
6,resnet50toponettau30,0.791071
7,cornets,0.788064
8,resnet101imagenet1kv1,0.782794
9,vitb14dinov2,0.772654


,model_type,model_name,probe_normed_pearsonr_median
0,best,vitb16siglip2res224,0.843869
1,median,vitb16synclr,0.733681
2,worst,alexnetipclref01,0.465497


In [16]:
probe_csv

,model_name,model_layer,model_froi,model_tthreshold,model_localizer,fmri_dataset,fmri_froi,fmri_tthreshold,normed_pearsonr_median,normed_pearsonr_mean,normed_pearsonr_sem,pearsonr_median,pearsonr_mean,pearsonr_sem,pearsonr,normed_pearsonr
0,alexnetimagenet1kv1,features.11,FFA,7,grillspector,nsd485,FFA,7,0.631920,0.640333,0.040214,0.464865,0.484840,0.038767,"[0.48758217692375183, 0.5925830602645874, 0.44...","[0.6000598561908851, 0.7411918583292154, 0.556..."
1,alexnetimagenet1kv1,features.11,FFA,7,grillspector,nsd485,OFA,7,0.606694,0.611209,0.015699,0.408266,0.423641,0.025031,"[0.42914092540740967, 0.38562384247779846, 0.4...","[0.6223200373575238, 0.5811705351905879, 0.650..."
2,alexnetimagenet1kv1,features.12,EBA,7,grillspector,nsd485,EBA,7,0.516884,0.512375,0.009396,0.398608,0.395944,0.011227,"[0.41173219680786133, 0.41715115308761597, 0.3...","[0.50853983942862, 0.5252288416404368, 0.48738..."
3,alexnetimagenet1kv1,features.12,EBA,7,grillspector,nsd485,FBA,7,0.532878,0.531079,0.024372,0.402158,0.379655,0.032716,"[0.42902088165283203, 0.41779109835624695, 0.3...","[0.5841787417948765, 0.556195395113791, 0.4743..."
4,alexnetimagenet1kv1,features.12,PPA,7,grillspector,nsd485,PPA,7,0.699738,0.686806,0.020156,0.509329,0.503079,0.031097,"[0.46852371096611023, 0.5501339435577393, 0.56...","[0.628036477907898, 0.7197126611218153, 0.6983..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,convnexttinyimagenet1kv1,features.5.7.block.3,EBA,7,grillspector,nsd485,EBA,7,0.837264,0.842066,0.015026,0.655607,0.650364,0.013322,"[0.6685240268707275, 0.6741549372673035, 0.642...","[0.8257092933581411, 0.8488185017978773, 0.812..."
248,convnexttinyimagenet1kv1,features.5.7.block.3,EBA,7,grillspector,nsd485,FBA,7,0.840972,0.849309,0.023229,0.642258,0.606639,0.045834,"[0.6707299947738647, 0.6310584545135498, 0.653...","[0.9133033406708395, 0.8401131757689446, 0.801..."
249,convnexttinyimagenet1kv1,features.5.2.block.3,PPA,7,grillspector,nsd485,PPA,7,0.870354,0.875082,0.020559,0.651226,0.640292,0.033450,"[0.6244229674339294, 0.7052236795425415, 0.678...","[0.8370129238141574, 0.9226087883384896, 0.844..."
250,convnexttinyimagenet1kv1,features.5.6.block.3,PPA,7,grillspector,nsd485,OPA,7,0.869323,0.869340,0.013877,0.514476,0.526575,0.029092,"[0.509183943271637, 0.5197678804397583, 0.6076...","[0.8515387742135094, 0.8983241831392043, 0.840..."


In [19]:
probe_csv_ = probe_csv[
    (probe_csv['fmri_froi'].isin(frois)) &
    (probe_csv['model_froi'].isin(frois)) &
    (probe_csv['fmri_tthreshold'] == tthreshold) &
    (probe_csv['model_tthreshold'] == tthreshold) &
    (probe_csv['model_localizer'] == model_localizer)
]
# Group by model name, get different columns for each froi's layer and its normed_pearsonr_median
froi_data = {}
for froi in frois:
    # Get data for this specific fROI
    froi_df = probe_csv_[(probe_csv_['fmri_froi'] == froi) & (probe_csv_['model_froi'] == froi)]
    # Group by model_name and get best layer and its score
    best_layers = froi_df.groupby('model_name').apply(
        lambda x: pd.Series({
            f'{froi}_layer': x.loc[x['normed_pearsonr_median'].idxmax(), 'model_layer'],
            f'{froi}_score': x['normed_pearsonr_median'].max()
        })
    )
    froi_data[froi] = best_layers

# Combine all fROI data
result = pd.DataFrame()
for froi, data in froi_data.items():
    if result.empty:
        result = data
    else:
        result = result.join(data)

# Reset index to make model_name a column
probe_csv_ = result.reset_index()

# Sort by average score across fROIs
probe_csv_['median_score'] = probe_csv_[[f'{froi}_score' for froi in frois]].median(axis=1)
probe_csv_['avg_score'] = probe_csv_[[f'{froi}_score' for froi in frois]].mean(axis=1)
sorted_indices = probe_csv_['median_score'].argsort()[::-1]  # Sort in descending order
display(probe_csv_.iloc[sorted_indices])

/tmp/ipykernel_555656/3153006206.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_layers = froi_df.groupby('model_name').apply(
/tmp/ipykernel_555656/3153006206.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_layers = froi_df.groupby('model_name').apply(
/tmp/ipykernel_555656/3153006206.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is de

,model_name,FFA_layer,FFA_score,PPA_layer,PPA_score,EBA_layer,EBA_score,median_score,avg_score
29,vitb16siglip2res224,trunk.blocks.8.norm1,0.779539,trunk.blocks.6.mlp,0.889988,trunk.blocks.6.attn,0.843869,0.843869,0.837799
18,resnet50openaiclip,layer4.1.relu1,0.838861,layer4.2.relu3,0.886035,layer4.0.bn3,0.777079,0.838861,0.833992
4,convnexttinyimagenet1kv1,features.5.7.block.3,0.692095,features.5.2.block.3,0.870354,features.5.7.block.3,0.837264,0.837264,0.799904
8,resnet101openaiclip,layer4.1.relu1,0.837051,layer3.11.conv1,0.900545,layer3.11.bn3,0.836885,0.837051,0.858160
25,vitb14dinov2reg,blocks.8.attn,0.712377,blocks.7.mlp,0.895587,blocks.9.norm2,0.831582,0.831582,0.813182
20,resnet50simclr,avgpool,0.691593,layer3.5.conv1,0.894144,avgpool,0.817388,0.817388,0.801042
22,resnet50toponettau30,layer4.1.relu3,0.768235,avgpool,0.842057,avgpool,0.791071,0.791071,0.800454
5,cornets,V4.nonlin2,0.630039,V4.norm1_3,0.850759,IT.norm3_0,0.788064,0.788064,0.756287
7,resnet101imagenet1kv1,layer3.22.relu1,0.651243,layer3.14.conv2,0.855994,avgpool,0.782794,0.782794,0.763344
24,vitb14dinov2,blocks.11.norm2,0.772654,blocks.10.norm2,0.901584,blocks.11.attn,0.747729,0.772654,0.807322


In [27]:
crossval_csv_selected = crossval_csv[
    (crossval_csv['fmri_froi'].isin(frois)) &
    (crossval_csv['model_froi'].isin(frois)) &
    (crossval_csv['fmri_tthreshold'] == tthreshold) &
    (crossval_csv['model_tthreshold'] == tthreshold) &
    (crossval_csv['model_localizer'] == model_localizer)
]
# Group by model_name, model_froi and fmri_froi and get the median of normed_pearsonr and normed_rdmr (across subjects)
crossval_csv_selected = crossval_csv_selected.groupby(['model_name', 'model_froi', 'fmri_froi'])[['normed_pearsonr', 'normed_rdmr']].median().reset_index()
# Group by the model_name and get the median of normed_pearsonr and normed_rdmr (across fROIs)
crossval_csv_selected = crossval_csv_selected.groupby(['model_name'])[['normed_pearsonr', 'normed_rdmr']].median().reset_index()
# Rename the columns to indicate that these are cross-validated metrics
crossval_csv_selected = crossval_csv_selected.rename(columns={
    'normed_pearsonr': 'crossval_normed_pearsonr',
    'normed_rdmr': 'crossval_normed_rdmr',
})
# Get the rank of all models in crossval_csv_selected separately for normed_pearsonr and normed_rdmr
crossval_csv_selected['crossval_rank_normed_pearsonr'] = crossval_csv_selected['crossval_normed_pearsonr'].rank(ascending=False, method='min')
crossval_csv_selected['crossval_rank_normed_rdmr'] = crossval_csv_selected['crossval_normed_rdmr'].rank(ascending=False, method='min')
# Merge models_df with crossval_csv_selected on model_name to get the cross-validated metrics for the best, median, and worst models
models_df_ = models_df.merge(crossval_csv_selected, on='model_name', how='left')
# Drop probe_normed_pearsonr_median, crossval_normed_pearsonr and crossval_normed_rdmr columns
# models_df_ = models_df_.drop(columns=['probe_normed_pearsonr_median', 'crossval_normed_pearsonr', 'crossval_normed_rdmr'])
display(models_df_)

,model_type,model_name,probe_normed_pearsonr_median,crossval_normed_pearsonr,crossval_normed_rdmr,crossval_rank_normed_pearsonr,crossval_rank_normed_rdmr
0,best,vitb16siglip2res224,0.843869,0.711801,0.481560,8.0,6.0
1,median,vitb16synclr,0.733681,0.696896,0.305190,12.0,27.0
2,worst,alexnetipclref01,0.465497,0.459088,0.268406,35.0,31.0


# PPA

In [4]:
froi = 'PPA'
probe_csv_selected = probe_csv[
    (probe_csv['fmri_froi'] == froi) &
    (probe_csv['model_froi'] == froi) &
    (probe_csv['fmri_tthreshold'] == tthreshold) &
    (probe_csv['model_tthreshold'] == tthreshold) &
    (probe_csv['model_localizer'] == model_localizer)
]
# Remove the row corresponding to model name resnet50imagenet21k
probe_csv_selected = probe_csv_selected[probe_csv_selected['model_name'] != 'resnet50imagenet21k']
# Group by the model_name and take the median of normed_pearsonr_median, ignore non-matching layers
# probe_csv_selected = probe_csv_selected.drop(columns=['model_layer'])
probe_csv_selected = probe_csv_selected.groupby('model_name')['normed_pearsonr_median'].median().reset_index()
# Sort the DataFrame by normed_pearsonr_median in descending order, for visualization purposes
probe_csv_selected = probe_csv_selected.sort_values(by='normed_pearsonr_median', ascending=False).reset_index(drop=True)
# Get the best, median, and worst model for this fROI based on normed_pearsonr_median
best_model_info = probe_csv_selected.loc[probe_csv_selected['normed_pearsonr_median'].idxmax()]
worst_model_info = probe_csv_selected.loc[probe_csv_selected['normed_pearsonr_median'].idxmin()]
# Find the median model using argsort
sorted_indices = np.argsort(probe_csv_selected['normed_pearsonr_median'].values)
median_idx = len(sorted_indices) // 2
median_model_info = probe_csv_selected.iloc[sorted_indices[median_idx]]

# Store the results in a DataFrame
models_df = [
    {
        'model_type': 'best',
        'model_name': best_model_info['model_name'],
        'probe_normed_pearsonr_median': best_model_info['normed_pearsonr_median'],
    }, 
    {
        'model_type': 'median',
        'model_name': median_model_info['model_name'],
        'probe_normed_pearsonr_median': median_model_info['normed_pearsonr_median'],
    },
    {
        'model_type': 'worst',
        'model_name': worst_model_info['model_name'],
        'probe_normed_pearsonr_median': worst_model_info['normed_pearsonr_median'],
    }
]
models_df = pd.DataFrame(models_df)
display(models_df)

,model_type,model_name,probe_normed_pearsonr_median
0,best,vitb14dinov2,0.901584
1,median,resnet50toponettau30,0.842057
2,worst,cornetz,0.625994


In [7]:
crossval_csv_selected = crossval_csv[
    (crossval_csv['fmri_froi'] == froi) &
    (crossval_csv['model_froi'] == froi) &
    (crossval_csv['fmri_tthreshold'] == tthreshold) &
    (crossval_csv['model_tthreshold'] == tthreshold) &
    (crossval_csv['model_localizer'] == model_localizer)
]
# Group by model_name, model_froi and fmri_froi and get the median of normed_pearsonr and normed_rdmr (across subjects)
crossval_csv_selected = crossval_csv_selected.groupby(['model_name', 'model_froi', 'fmri_froi'])[['normed_pearsonr', 'normed_rdmr']].median().reset_index()
# Group by the model_name and get the median of normed_pearsonr and normed_rdmr (across fROIs)
crossval_csv_selected = crossval_csv_selected.groupby(['model_name'])[['normed_pearsonr', 'normed_rdmr']].median().reset_index()
# Rename the columns to indicate that these are cross-validated metrics
crossval_csv_selected = crossval_csv_selected.rename(columns={
    'normed_pearsonr': 'crossval_normed_pearsonr',
    'normed_rdmr': 'crossval_normed_rdmr',
})
# Get the rank of all models in crossval_csv_selected separately for normed_pearsonr and normed_rdmr
crossval_csv_selected['crossval_rank_normed_pearsonr'] = crossval_csv_selected['crossval_normed_pearsonr'].rank(ascending=False, method='min')
crossval_csv_selected['crossval_rank_normed_rdmr'] = crossval_csv_selected['crossval_normed_rdmr'].rank(ascending=False, method='min')
# Merge models_df with crossval_csv_selected on model_name to get the cross-validated metrics for the best, median, and worst models
models_df_ = models_df.merge(crossval_csv_selected, on='model_name', how='left')
# Drop probe_normed_pearsonr_median, crossval_normed_pearsonr and crossval_normed_rdmr columns
# models_df_ = models_df_.drop(columns=['probe_normed_pearsonr_median', 'crossval_normed_pearsonr', 'crossval_normed_rdmr'])
display(models_df_)

,model_type,model_name,probe_normed_pearsonr_median,crossval_normed_pearsonr,crossval_normed_rdmr,crossval_rank_normed_pearsonr,crossval_rank_normed_rdmr
0,best,vitb14dinov2,0.901584,0.726980,0.433561,18.0,8.0
1,median,resnet50toponettau30,0.842057,0.753271,0.405016,11.0,11.0
2,worst,cornetz,0.625994,0.586357,0.271312,33.0,30.0


# EBA

In [3]:
froi = 'EBA'
probe_csv_selected = probe_csv[
    (probe_csv['fmri_froi'] == froi) &
    (probe_csv['model_froi'] == froi) &
    (probe_csv['fmri_tthreshold'] == tthreshold) &
    (probe_csv['model_tthreshold'] == tthreshold) &
    (probe_csv['model_localizer'] == model_localizer)
]
# Remove the row corresponding to model name resnet50imagenet21k
probe_csv_selected = probe_csv_selected[probe_csv_selected['model_name'] != 'resnet50imagenet21k']
# Group by the model_name and take the median of normed_pearsonr_median, ignore non-matching layers
# probe_csv_selected = probe_csv_selected.drop(columns=['model_layer'])
probe_csv_selected = probe_csv_selected.groupby('model_name')['normed_pearsonr_median'].median().reset_index()
# Sort the DataFrame by normed_pearsonr_median in descending order, for visualization purposes
probe_csv_selected = probe_csv_selected.sort_values(by='normed_pearsonr_median', ascending=False).reset_index(drop=True)
# Get the best, median, and worst model for this fROI based on normed_pearsonr_median
best_model_info = probe_csv_selected.loc[probe_csv_selected['normed_pearsonr_median'].idxmax()]
worst_model_info = probe_csv_selected.loc[probe_csv_selected['normed_pearsonr_median'].idxmin()]
# Find the median model using argsort
sorted_indices = np.argsort(probe_csv_selected['normed_pearsonr_median'].values)
median_idx = len(sorted_indices) // 2
median_model_info = probe_csv_selected.iloc[sorted_indices[median_idx]]

# Store the results in a DataFrame
models_df = [
    {
        'model_type': 'best',
        'model_name': best_model_info['model_name'],
        'probe_normed_pearsonr_median': best_model_info['normed_pearsonr_median'],
    }, 
    {
        'model_type': 'median',
        'model_name': median_model_info['model_name'],
        'probe_normed_pearsonr_median': median_model_info['normed_pearsonr_median'],
    },
    {
        'model_type': 'worst',
        'model_name': worst_model_info['model_name'],
        'probe_normed_pearsonr_median': worst_model_info['normed_pearsonr_median'],
    }
]
models_df = pd.DataFrame(models_df)
display(models_df)

,model_type,model_name,probe_normed_pearsonr_median
0,best,vitb16siglip2res224,0.843869
1,median,vitb16synclr,0.733681
2,worst,cornetz,0.327232


In [4]:
crossval_csv_selected = crossval_csv[
    (crossval_csv['fmri_froi'] == froi) &
    (crossval_csv['model_froi'] == froi) &
    (crossval_csv['fmri_tthreshold'] == tthreshold) &
    (crossval_csv['model_tthreshold'] == tthreshold) &
    (crossval_csv['model_localizer'] == model_localizer)
]
# Group by model_name, model_froi and fmri_froi and get the median of normed_pearsonr and normed_rdmr (across subjects)
crossval_csv_selected = crossval_csv_selected.groupby(['model_name', 'model_froi', 'fmri_froi'])[['normed_pearsonr', 'normed_rdmr']].median().reset_index()
# Group by the model_name and get the median of normed_pearsonr and normed_rdmr (across fROIs)
crossval_csv_selected = crossval_csv_selected.groupby(['model_name'])[['normed_pearsonr', 'normed_rdmr']].median().reset_index()
# Rename the columns to indicate that these are cross-validated metrics
crossval_csv_selected = crossval_csv_selected.rename(columns={
    'normed_pearsonr': 'crossval_normed_pearsonr',
    'normed_rdmr': 'crossval_normed_rdmr',
})
# Get the rank of all models in crossval_csv_selected separately for normed_pearsonr and normed_rdmr
crossval_csv_selected['crossval_rank_normed_pearsonr'] = crossval_csv_selected['crossval_normed_pearsonr'].rank(ascending=False, method='min')
crossval_csv_selected['crossval_rank_normed_rdmr'] = crossval_csv_selected['crossval_normed_rdmr'].rank(ascending=False, method='min')
# Merge models_df with crossval_csv_selected on model_name to get the cross-validated metrics for the best, median, and worst models
models_df_ = models_df.merge(crossval_csv_selected, on='model_name', how='left')
# Drop probe_normed_pearsonr_median, crossval_normed_pearsonr and crossval_normed_rdmr columns
# models_df_ = models_df_.drop(columns=['probe_normed_pearsonr_median', 'crossval_normed_pearsonr', 'crossval_normed_rdmr'])
display(models_df_)

,model_type,model_name,probe_normed_pearsonr_median,crossval_normed_pearsonr,crossval_normed_rdmr,crossval_rank_normed_pearsonr,crossval_rank_normed_rdmr
0,best,vitb16siglip2res224,0.843869,0.764139,0.481560,2.0,6.0
1,median,vitb16synclr,0.733681,0.697310,0.305190,11.0,19.0
2,worst,cornetz,0.327232,0.349487,0.210548,35.0,29.0
